In [0]:
%sql
SELECT * FROM workspace.bronze.erp_px_cat_g1v2raw

#Initialisations

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim, add_months, substring, to_date, regexp_replace
from pyspark.sql.types import StringType

In [0]:
RENAME_MAP = {
    'ID': 'id',
    'CAT': 'category',
    'SUBCAT': 'sub_category',
    'MAINTENANCE': 'maintenance'
}

#Read the bronze table in

In [0]:
df = spark.read.table("workspace.bronze.erp_px_cat_g1v2raw")
df.show()

#Transformations

##1. Trimming all text column to remove whitespaces

In [0]:
# for field in df.schema:
#     if isinstance(field.dataType, StringType):
#         df = df.withColumn(field.name, trim(col(field.name)))

df = df.withColumns({
    feild.name: trim(col(feild.name))
    for feild in df.schema 
    if isinstance(feild.dataType, StringType)
})
df.show()

##2. Preparing the keys for modelling

In [0]:
df = df.withColumn("sub_category_key", regexp_replace(col("ID"), "_", "-"))
df.show()

##3. Rename columns to business friendly names

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
df.show() 

#Writting the dataframe to silver layer

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.erp_products")
)

In [0]:
%sql
SELECT * FROM silver.erp_products